In [2]:

import ccxt
import time
import pandas as pd
from datetime import datetime

# ==================== KONFIGURATION ====================
# Deine MEXC API Credentials (NIEMALS öffentlich teilen!)
API_KEY = ''
API_SECRET = ''

# Handelsparameter
SYMBOL = 'UCN/USDT'  # Handelspaar
POSITION_SIZE_USDT = 10  # Wie viel USDT pro Trade investiert werden soll
TARGET_PROFIT_PERCENT = 2.0  # Gewinnziel in Prozent (z.B. 2.0 = 2%)
STOP_LOSS_PERCENT = 1.0  # Stop-Loss in Prozent (z.B. 1.0 = 1% Verlust)
CHECK_INTERVAL = 5  # Wie oft in Sekunden der Preis überprüft wird
MAX_POSITIONS = 1  # Maximale Anzahl gleichzeitiger Positionen

# ==================== EXCHANGE SETUP ====================
exchange = ccxt.mexc({
    'apiKey': API_KEY,
    'secret': API_SECRET,
    'enableRateLimit': True,
    'options': {
        'defaultType': 'spot',  # 'spot' für Spot-Trading oder 'future' für Futures
    }
})

# ==================== HILFSFUNKTIONEN ====================

def get_current_price(symbol):
    """
    Holt den aktuellen Preis für das angegebene Symbol
    """
    ticker = exchange.fetch_ticker(symbol)
    return ticker['last']

def get_balance(currency='USDT'):
    """
    Holt das verfügbare Guthaben für eine Währung
    """
    balance = exchange.fetch_balance()
    return balance['free'][currency]

def calculate_quantity(price, usdt_amount):
    """
    Berechnet die Menge an Coins basierend auf USDT-Betrag und aktuellem Preis
    """
    quantity = usdt_amount / price
    # Runde auf die von MEXC akzeptierte Genauigkeit (meistens 2-8 Dezimalstellen)
    return round(quantity, 6)

def place_buy_order(symbol, quantity):
    """
    Platziert eine Market-Kauforder
    """
    try:
        order = exchange.create_market_buy_order(symbol, quantity)
        print(f"Kauforder platziert: {quantity} {symbol}")
        print(f"Order ID: {order['id']}")
        return order
    except Exception as e:
        print(f"Fehler beim Kaufen: {e}")
        return None

def place_sell_order(symbol, quantity):
    """
    Platziert eine Market-Verkaufsorder
    """
    try:
        order = exchange.create_market_sell_order(symbol, quantity)
        print(f"Verkaufsorder platziert: {quantity} {symbol}")
        print(f"Order ID: {order['id']}")
        return order
    except Exception as e:
        print(f"Fehler beim Verkaufen: {e}")
        return None

def log_trade(action, price, quantity, profit=None):
    """
    Loggt einen Trade mit Zeitstempel
    """
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    profit_str = f" | Gewinn: {profit:.2f}%" if profit else ""
    print(f"[{timestamp}] {action} | Preis: {price:.6f} | Menge: {quantity:.6f}{profit_str}")

# ==================== HAUPTBOT-LOGIK ====================

class TradingBot:
    def __init__(self):
        self.active_positions = []  # Liste aktiver Positionen: [{'entry_price': float, 'quantity': float, 'target': float, 'stop_loss': float}]
        
    def open_position(self):
        """
        Öffnet eine neue Long-Position
        """
        if len(self.active_positions) >= MAX_POSITIONS:
            print(f"Maximale Anzahl von Positionen ({MAX_POSITIONS}) erreicht")
            return False
        
        try:
            # Aktuellen Preis holen
            current_price = get_current_price(SYMBOL)
            
            # Menge berechnen
            quantity = calculate_quantity(current_price, POSITION_SIZE_USDT)
            
            # Kauforder platzieren
            order = place_buy_order(SYMBOL, quantity)
            
            if order:
                # Zielpreis und Stop-Loss berechnen
                target_price = current_price * (1 + TARGET_PROFIT_PERCENT / 100)
                stop_loss_price = current_price * (1 - STOP_LOSS_PERCENT / 100)
                
                # Position zur Liste hinzufügen
                position = {
                    'entry_price': current_price,
                    'quantity': quantity,
                    'target': target_price,
                    'stop_loss': stop_loss_price,
                    'entry_time': datetime.now()
                }
                self.active_positions.append(position)
                
                log_trade("KAUF", current_price, quantity)
                print(f"Zielpreis: {target_price:.6f} (+{TARGET_PROFIT_PERCENT}%)")
                print(f"Stop-Loss: {stop_loss_price:.6f} (-{STOP_LOSS_PERCENT}%)")
                
                return True
        except Exception as e:
            print(f"Fehler beim Öffnen der Position: {e}")
        
        return False
    
    def check_positions(self):
        """
        Überprüft alle aktiven Positionen und schließt sie bei Erreichen des Ziels oder Stop-Loss
        """
        if not self.active_positions:
            return
        
        current_price = get_current_price(SYMBOL)
        positions_to_close = []
        
        for i, pos in enumerate(self.active_positions):
            # Prüfe ob Zielpreis erreicht wurde
            if current_price >= pos['target']:
                profit_percent = ((current_price - pos['entry_price']) / pos['entry_price']) * 100
                print(f"\n🎉 ZIELPREIS ERREICHT!")
                
                if place_sell_order(SYMBOL, pos['quantity']):
                    log_trade("VERKAUF (GEWINN)", current_price, pos['quantity'], profit_percent)
                    positions_to_close.append(i)
            
            # Prüfe ob Stop-Loss erreicht wurde
            elif current_price <= pos['stop_loss']:
                loss_percent = ((current_price - pos['entry_price']) / pos['entry_price']) * 100
                print(f"\n STOP-LOSS AUSGELÖST!")
                
                if place_sell_order(SYMBOL, pos['quantity']):
                    log_trade("VERKAUF (VERLUST)", current_price, pos['quantity'], loss_percent)
                    positions_to_close.append(i)
        
        # Geschlossene Positionen aus der Liste entfernen
        for i in sorted(positions_to_close, reverse=True):
            self.active_positions.pop(i)
    
    def display_status(self):
        """
        Zeigt den aktuellen Status des Bots an
        """
        current_price = get_current_price(SYMBOL)
        print(f"\n{'='*60}")
        print(f"Aktueller Preis: {current_price:.6f} {SYMBOL}")
        print(f"Aktive Positionen: {len(self.active_positions)}/{MAX_POSITIONS}")
        
        for i, pos in enumerate(self.active_positions, 1):
            profit_percent = ((current_price - pos['entry_price']) / pos['entry_price']) * 100
            print(f"\n   Position {i}:")
            print(f"   Einstieg: {pos['entry_price']:.6f} | Aktuell: {profit_percent:+.2f}%")
            print(f"   Ziel: {pos['target']:.6f} | Stop: {pos['stop_loss']:.6f}")
        print(f"{'='*60}")
    
    def run(self):
        """
        Hauptschleife des Trading Bots
        """
        print("\n MEXC Trading Bot gestartet")
        print(f" Symbol: {SYMBOL}")
        print(f" Position Size: {POSITION_SIZE_USDT} USDT")
        print(f" Gewinnziel: {TARGET_PROFIT_PERCENT}%")
        print(f" Stop-Loss: {STOP_LOSS_PERCENT}%")
        print(f"⏱ Check Interval: {CHECK_INTERVAL}s\n")
        
        try:
            while True:
                # Status anzeigen
                self.display_status()
                
                # Positionen überprüfen
                self.check_positions()
                
                # Optional: Neue Position öffnen wenn keine aktiv ist
                # Kommentiere die nächsten 2 Zeilen aus, wenn du Positionen manuell öffnen willst
                # if len(self.active_positions) == 0:
                #     self.open_position()
                
                # Warten bis zur nächsten Überprüfung
                time.sleep(CHECK_INTERVAL)
                
        except KeyboardInterrupt:
            print("\n\n Bot wurde gestoppt")
            print(f"Offene Positionen: {len(self.active_positions)}")

# ==================== BOT STARTEN ====================
if __name__ == "__main__":
    bot = TradingBot()
    
    # Öffne eine initiale Position
    bot.open_position()
    
    # Starte die Überwachungsschleife
    bot.run()

Kauforder platziert: 0.006199 UCN/USDT
Order ID: C02__623574158768504832016
[2025-11-29 23:04:58] KAUF | Preis: 1613.140000 | Menge: 0.006199
Zielpreis: 1645.402800 (+2.0%)
Stop-Loss: 1597.008600 (-1.0%)

 MEXC Trading Bot gestartet
 Symbol: UCN/USDT
 Position Size: 10 USDT
 Gewinnziel: 2.0%
 Stop-Loss: 1.0%
⏱ Check Interval: 5s


Aktueller Preis: 1613.140000 UCN/USDT
Aktive Positionen: 1/1

   Position 1:
   Einstieg: 1613.140000 | Aktuell: +0.00%
   Ziel: 1645.402800 | Stop: 1597.008600

Aktueller Preis: 1613.140000 UCN/USDT
Aktive Positionen: 1/1

   Position 1:
   Einstieg: 1613.140000 | Aktuell: +0.00%
   Ziel: 1645.402800 | Stop: 1597.008600

Aktueller Preis: 1613.140000 UCN/USDT
Aktive Positionen: 1/1

   Position 1:
   Einstieg: 1613.140000 | Aktuell: +0.00%
   Ziel: 1645.402800 | Stop: 1597.008600

Aktueller Preis: 1613.150000 UCN/USDT
Aktive Positionen: 1/1

   Position 1:
   Einstieg: 1613.140000 | Aktuell: +0.00%
   Ziel: 1645.402800 | Stop: 1597.008600

Aktueller Preis: 16

In [3]:
# Zelle 1: Installationen und Imports
# Führe diese Zelle einmal aus, um alle benötigten Bibliotheken zu installieren
# !pip install ccxt pandas

import ccxt
import time
import pandas as pd
from datetime import datetime
from IPython.display import clear_output

print("✅ Alle Bibliotheken erfolgreich importiert!")

# ==================================================================================
# Zelle 2: KONFIGURATION - Passe hier alle Parameter an
# ==================================================================================

# ==================== API CREDENTIALS ====================
# Trage hier deine MEXC API Keys ein
API_KEY = ''
API_SECRET = ''

# ==================== HANDELSPARAMETER ====================
SYMBOL = 'UCN/USDT'  # Handelspaar (welche Kryptowährung?)
POSITION_SIZE_USDT = 100  # Wie viel USDT pro Trade investiert werden soll

# ==================== ORDER-TYP AUSWAHL ====================
# 'market' = Sofortige Ausführung zum aktuellen Preis (schnell, aber evtl. ungünstiger Preis)
# 'limit' = Order wird nur zum gewünschten Preis ausgeführt (besser kontrollierbar)
ORDER_TYPE = 'limit'  # Ändere zu 'market' für Market-Orders

# Nur für Limit-Orders relevant:
# Wie weit vom aktuellen Preis soll die Kauf-Order entfernt sein?
# 0.1 = 0.1% unter dem aktuellen Preis (günstiger einkaufen)
BUY_LIMIT_OFFSET_PERCENT = 0.1

# ==================== GEWINN- UND VERLUST-ZIELE ====================
TARGET_PROFIT_PERCENT = 2.0  # Gewinnziel in % (z.B. 2.0 = verkaufe bei +2% Gewinn)
STOP_LOSS_PERCENT = 1.0  # Stop-Loss in % (z.B. 1.0 = verkaufe bei -1% Verlust)

# ==================== BOT-VERHALTEN ====================
CHECK_INTERVAL = 10  # Wie oft in Sekunden der Preis überprüft wird
MAX_POSITIONS = 1  # Maximale Anzahl gleichzeitiger Positionen
AUTO_PLACE_SELL_ORDERS = True  # Soll der Bot automatisch Verkaufs-Limit-Orders setzen?

# ==================== ANZEIGE-EINSTELLUNGEN ====================
SHOW_DETAILED_LOGS = True  # Detaillierte Logs anzeigen?

print("✅ Konfiguration geladen!")
print(f"📊 Symbol: {SYMBOL}")
print(f"💰 Position Size: {POSITION_SIZE_USDT} USDT")
print(f"📈 Order Type: {ORDER_TYPE.upper()}")
print(f"🎯 Gewinnziel: {TARGET_PROFIT_PERCENT}%")
print(f"🛑 Stop-Loss: {STOP_LOSS_PERCENT}%")

# ==================================================================================
# Zelle 3: Exchange-Verbindung einrichten
# ==================================================================================

# Verbindung zur MEXC Exchange herstellen
exchange = ccxt.mexc({
    'apiKey': API_KEY,
    'secret': API_SECRET,
    'enableRateLimit': True,  # Schützt vor zu vielen Anfragen
    'options': {
        'defaultType': 'spot',  # Spot-Trading (keine Hebelwirkung)
    }
})

# Teste die Verbindung
try:
    balance = exchange.fetch_balance()
    print("✅ Verbindung zu MEXC erfolgreich!")
    print(f"💵 USDT Guthaben: {balance['free'].get('USDT', 0):.2f}")

    # Zeige aktuellen Preis
    ticker = exchange.fetch_ticker(SYMBOL)
    print(f"💹 Aktueller {SYMBOL} Preis: {ticker['last']:.4f}")

except Exception as e:
    print(f"❌ Fehler bei der Verbindung: {e}")
    print("⚠️ Bitte überprüfe deine API Keys!")

# ==================================================================================
# Zelle 4: Hilfsfunktionen definieren
# ==================================================================================

def get_current_price(symbol):
    """
    Holt den aktuellen Marktpreis für ein Symbol
    """
    ticker = exchange.fetch_ticker(symbol)
    return ticker['last']

def get_balance(currency='USDT'):
    """
    Holt das verfügbare Guthaben für eine Währung
    """
    balance = exchange.fetch_balance()
    return balance['free'].get(currency, 0)

def calculate_quantity(price, usdt_amount):
    """
    Berechnet die Menge an Coins basierend auf USDT-Betrag und Preis
    Rundet auf die von MEXC akzeptierte Genauigkeit
    """
    quantity = usdt_amount / price
    return round(quantity, 6)

def place_buy_order(symbol, quantity, order_type='market', limit_price=None):
    """
    Platziert eine Kauforder

    order_type: 'market' oder 'limit'
    limit_price: Preis für Limit-Order (nur bei order_type='limit')
    """
    try:
        if order_type == 'market':
            order = exchange.create_market_buy_order(symbol, quantity)
            print(f"✅ MARKET-Kauforder ausgeführt: {quantity} {symbol}")
        else:  # limit
            if limit_price is None:
                raise ValueError("Limit-Preis muss angegeben werden!")
            order = exchange.create_limit_buy_order(symbol, quantity, limit_price)
            print(f"✅ LIMIT-Kauforder platziert: {quantity} {symbol} @ {limit_price:.6f}")
            print(f"   ⏳ Order wartet auf Ausführung...")

        if SHOW_DETAILED_LOGS:
            print(f"   📋 Order ID: {order['id']}")
            print(f"   💵 Order Amount: ~{quantity * (limit_price if limit_price else get_current_price(symbol)):.2f} USDT")

        return order
    except Exception as e:
        print(f"❌ Fehler beim Kaufen: {e}")
        return None

def place_sell_order(symbol, quantity, order_type='market', limit_price=None):
    """
    Platziert eine Verkaufsorder

    order_type: 'market' oder 'limit'
    limit_price: Preis für Limit-Order (nur bei order_type='limit')
    """
    try:
        if order_type == 'market':
            order = exchange.create_market_sell_order(symbol, quantity)
            print(f"✅ MARKET-Verkaufsorder ausgeführt: {quantity} {symbol}")
        else:  # limit
            if limit_price is None:
                raise ValueError("Limit-Preis muss angegeben werden!")
            order = exchange.create_limit_sell_order(symbol, quantity, limit_price)
            print(f"✅ LIMIT-Verkaufsorder platziert: {quantity} {symbol} @ {limit_price:.6f}")
            print(f"   ⏳ Order wartet auf Ausführung...")

        if SHOW_DETAILED_LOGS:
            print(f"   📋 Order ID: {order['id']}")

        return order
    except Exception as e:
        print(f"❌ Fehler beim Verkaufen: {e}")
        return None

def cancel_order(symbol, order_id):
    """
    Storniert eine offene Order
    """
    try:
        exchange.cancel_order(order_id, symbol)
        print(f"🗑️ Order {order_id} storniert")
        return True
    except Exception as e:
        print(f"❌ Fehler beim Stornieren: {e}")
        return False

def get_open_orders(symbol):
    """
    Holt alle offenen Orders für ein Symbol
    """
    try:
        return exchange.fetch_open_orders(symbol)
    except Exception as e:
        print(f"❌ Fehler beim Abrufen offener Orders: {e}")
        return []

def log_trade(action, price, quantity, profit=None):
    """
    Loggt einen Trade mit Zeitstempel
    """
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    profit_str = f" | 📊 P&L: {profit:+.2f}%" if profit else ""
    print(f"[{timestamp}] {action} | 💰 Preis: {price:.6f} | 📦 Menge: {quantity:.6f}{profit_str}")

print("✅ Hilfsfunktionen definiert!")

# ==================================================================================
# Zelle 5: Trading Bot Klasse
# ==================================================================================

class TradingBot:
    """
    Hauptklasse für den Trading Bot
    Verwaltet Positionen, Orders und überwacht den Markt
    """

    def __init__(self):
        self.active_positions = []  # Liste der aktiven Positionen
        self.trade_history = []  # Historie aller Trades
        self.sell_orders = {}  # Tracking von Verkaufs-Orders {order_id: position_index}

    def open_position(self):
        """
        Öffnet eine neue Long-Position
        """
        if len(self.active_positions) >= MAX_POSITIONS:
            print(f"⚠️ Maximale Anzahl von Positionen ({MAX_POSITIONS}) erreicht")
            return False

        try:
            # Aktuellen Preis holen
            current_price = get_current_price(SYMBOL)
            print(f"\n{'='*60}")
            print(f"🔍 Öffne neue Position...")
            print(f"💹 Aktueller Marktpreis: {current_price:.6f}")

            # Kaufpreis und Menge berechnen
            if ORDER_TYPE == 'limit':
                # Bei Limit-Order: Kaufe etwas unter dem aktuellen Preis
                buy_price = current_price * (1 - BUY_LIMIT_OFFSET_PERCENT / 100)
                print(f"🎯 Limit-Kaufpreis: {buy_price:.6f} (-{BUY_LIMIT_OFFSET_PERCENT}%)")
            else:
                buy_price = current_price

            quantity = calculate_quantity(buy_price, POSITION_SIZE_USDT)

            # Kauforder platzieren
            order = place_buy_order(SYMBOL, quantity, ORDER_TYPE, buy_price if ORDER_TYPE == 'limit' else None)

            if order:
                # Zielpreise berechnen
                target_price = buy_price * (1 + TARGET_PROFIT_PERCENT / 100)
                stop_loss_price = buy_price * (1 - STOP_LOSS_PERCENT / 100)

                # Position erstellen
                position = {
                    'entry_price': buy_price,
                    'quantity': quantity,
                    'target': target_price,
                    'stop_loss': stop_loss_price,
                    'entry_time': datetime.now(),
                    'buy_order_id': order['id'],
                    'buy_order_status': 'open' if ORDER_TYPE == 'limit' else 'filled',
                    'sell_order_id': None
                }

                self.active_positions.append(position)

                log_trade("🟢 KAUF-ORDER", buy_price, quantity)
                print(f"   🎯 Zielpreis (Verkauf): {target_price:.6f} (+{TARGET_PROFIT_PERCENT}%)")
                print(f"   🛑 Stop-Loss: {stop_loss_price:.6f} (-{STOP_LOSS_PERCENT}%)")

                # Bei Market-Orders: Sofort Verkaufs-Limit-Order setzen
                if ORDER_TYPE == 'market' and AUTO_PLACE_SELL_ORDERS:
                    self.place_take_profit_order(len(self.active_positions) - 1)

                print(f"{'='*60}\n")
                return True

        except Exception as e:
            print(f"❌ Fehler beim Öffnen der Position: {e}")

        return False

    def place_take_profit_order(self, position_index):
        """
        Setzt eine Limit-Verkaufsorder zum Zielpreis
        Diese Order bleibt auf MEXC aktiv, auch wenn der Bot stoppt!
        """
        pos = self.active_positions[position_index]

        try:
            # Verkaufs-Limit-Order zum Zielpreis platzieren
            sell_order = place_sell_order(
                SYMBOL,
                pos['quantity'],
                'limit',
                pos['target']
            )

            if sell_order:
                pos['sell_order_id'] = sell_order['id']
                self.sell_orders[sell_order['id']] = position_index
                print(f"   ✅ Verkaufs-Order aktiv - bleibt auf MEXC bestehen!")
                return True

        except Exception as e:
            print(f"   ⚠️ Konnte Verkaufs-Order nicht setzen: {e}")

        return False

    def check_buy_orders(self):
        """
        Prüft ob Limit-Kauforders ausgeführt wurden
        """
        for i, pos in enumerate(self.active_positions):
            if pos['buy_order_status'] == 'open':
                try:
                    # Prüfe Order-Status
                    order = exchange.fetch_order(pos['buy_order_id'], SYMBOL)

                    if order['status'] == 'closed':
                        print(f"\n✅ Kauforder ausgeführt!")
                        pos['buy_order_status'] = 'filled'
                        pos['entry_price'] = order['average']  # Tatsächlicher Kaufpreis

                        # Zielpreise neu berechnen
                        pos['target'] = pos['entry_price'] * (1 + TARGET_PROFIT_PERCENT / 100)
                        pos['stop_loss'] = pos['entry_price'] * (1 - STOP_LOSS_PERCENT / 100)

                        # Verkaufs-Order setzen
                        if AUTO_PLACE_SELL_ORDERS:
                            self.place_take_profit_order(i)

                except Exception as e:
                    if SHOW_DETAILED_LOGS:
                        print(f"⚠️ Fehler beim Prüfen der Kauforder: {e}")

    def check_sell_orders(self):
        """
        Prüft ob Verkaufs-Limit-Orders ausgeführt wurden
        """
        completed_positions = []

        for order_id, pos_index in list(self.sell_orders.items()):
            if pos_index >= len(self.active_positions):
                continue

            pos = self.active_positions[pos_index]

            try:
                order = exchange.fetch_order(order_id, SYMBOL)

                if order['status'] == 'closed':
                    print(f"\n🎉 VERKAUFS-ORDER AUSGEFÜHRT!")
                    sell_price = order['average']
                    profit_percent = ((sell_price - pos['entry_price']) / pos['entry_price']) * 100

                    log_trade("🟢 VERKAUF (GEWINN)", sell_price, pos['quantity'], profit_percent)

                    # Trade zur Historie hinzufügen
                    self.trade_history.append({
                        'entry_price': pos['entry_price'],
                        'exit_price': sell_price,
                        'quantity': pos['quantity'],
                        'profit_percent': profit_percent,
                        'profit_usdt': (sell_price - pos['entry_price']) * pos['quantity'],
                        'timestamp': datetime.now()
                    })

                    completed_positions.append(pos_index)
                    del self.sell_orders[order_id]

            except Exception as e:
                if SHOW_DETAILED_LOGS:
                    print(f"⚠️ Fehler beim Prüfen der Verkaufsorder: {e}")

        # Abgeschlossene Positionen entfernen
        for i in sorted(completed_positions, reverse=True):
            self.active_positions.pop(i)

    def check_positions(self):
        """
        Überwacht aktive Positionen (nur für manuelles Trading ohne Auto-Sell-Orders)
        """
        if AUTO_PLACE_SELL_ORDERS:
            # Bei Auto-Sell-Orders nur Orders prüfen
            self.check_buy_orders()
            self.check_sell_orders()
            return

        # Manuelles Monitoring (ohne Auto-Sell)
        current_price = get_current_price(SYMBOL)
        positions_to_close = []

        for i, pos in enumerate(self.active_positions):
            if pos['buy_order_status'] != 'filled':
                continue

            # Prüfe Zielpreis
            if current_price >= pos['target']:
                profit_percent = ((current_price - pos['entry_price']) / pos['entry_price']) * 100
                print(f"\n🎉 ZIELPREIS ERREICHT!")

                if place_sell_order(SYMBOL, pos['quantity'], 'market'):
                    log_trade("🟢 VERKAUF (GEWINN)", current_price, pos['quantity'], profit_percent)
                    positions_to_close.append(i)

            # Prüfe Stop-Loss
            elif current_price <= pos['stop_loss']:
                loss_percent = ((current_price - pos['entry_price']) / pos['entry_price']) * 100
                print(f"\n🛑 STOP-LOSS AUSGELÖST!")

                if place_sell_order(SYMBOL, pos['quantity'], 'market'):
                    log_trade("🔴 VERKAUF (VERLUST)", current_price, pos['quantity'], loss_percent)
                    positions_to_close.append(i)

        # Geschlossene Positionen entfernen
        for i in sorted(positions_to_close, reverse=True):
            self.active_positions.pop(i)

    def display_status(self):
        """
        Zeigt den aktuellen Status des Bots
        """
        current_price = get_current_price(SYMBOL)

        print(f"\n{'='*70}")
        print(f"🤖 TRADING BOT STATUS | {datetime.now().strftime('%H:%M:%S')}")
        print(f"{'='*70}")
        print(f"💹 Aktueller {SYMBOL} Preis: {current_price:.6f}")
        print(f"📊 Aktive Positionen: {len(self.active_positions)}/{MAX_POSITIONS}")
        print(f"💵 USDT Guthaben: {get_balance('USDT'):.2f}")

        if self.active_positions:
            print(f"\n{'─'*70}")
            for i, pos in enumerate(self.active_positions, 1):
                status = "⏳ Warte auf Ausführung" if pos['buy_order_status'] == 'open' else "✅ Aktiv"
                print(f"\n📍 Position {i} - {status}")
                print(f"   Einstieg: {pos['entry_price']:.6f} USDT")

                if pos['buy_order_status'] == 'filled':
                    profit_percent = ((current_price - pos['entry_price']) / pos['entry_price']) * 100
                    profit_usdt = (current_price - pos['entry_price']) * pos['quantity']
                    emoji = "🟢" if profit_percent > 0 else "🔴"
                    print(f"   Aktuell: {emoji} {profit_percent:+.2f}% ({profit_usdt:+.2f} USDT)")
                    print(f"   🎯 Ziel: {pos['target']:.6f} (+{TARGET_PROFIT_PERCENT}%)")
                    print(f"   🛑 Stop: {pos['stop_loss']:.6f} (-{STOP_LOSS_PERCENT}%)")

                    if pos['sell_order_id']:
                        print(f"   📝 Verkaufs-Order aktiv auf MEXC")

        if self.trade_history:
            total_profit = sum(t['profit_usdt'] for t in self.trade_history)
            print(f"\n{'─'*70}")
            print(f"📈 Abgeschlossene Trades: {len(self.trade_history)}")
            print(f"💰 Gesamtgewinn/-verlust: {total_profit:+.2f} USDT")

        print(f"{'='*70}\n")

    def run(self):
        """
        Hauptschleife des Trading Bots
        """
        print("\n🚀 MEXC TRADING BOT GESTARTET")
        print(f"📊 Order-Typ: {ORDER_TYPE.upper()}")
        print(f"🎯 Auto-Verkauf: {'AN' if AUTO_PLACE_SELL_ORDERS else 'AUS'}")
        print(f"⏱️ Check-Interval: {CHECK_INTERVAL}s\n")

        try:
            iteration = 0
            while True:
                iteration += 1

                # Status anzeigen (alle 3 Iterationen für bessere Lesbarkeit)
                if iteration % 3 == 1:
                    self.display_status()
                else:
                    print(f"⏳ Überwache Positionen... [{datetime.now().strftime('%H:%M:%S')}]")

                # Positionen prüfen
                self.check_positions()

                # Warten
                time.sleep(CHECK_INTERVAL)

        except KeyboardInterrupt:
            print("\n\n⏹️ BOT GESTOPPT")
            self.display_status()
            print("\n⚠️ WICHTIG:")
            if AUTO_PLACE_SELL_ORDERS:
                print("✅ Verkaufs-Orders bleiben auf MEXC aktiv!")
                print("   Sie werden automatisch ausgeführt, auch wenn der Bot aus ist.")
            else:
                print("⚠️ Du musst jetzt manuell verkaufen oder den Bot neu starten!")

print("✅ Trading Bot Klasse definiert!")

# ==================================================================================
# Zelle 6: Bot starten - HAUPTZELLE ZUM AUSFÜHREN
# ==================================================================================

# Erstelle Bot-Instanz
bot = TradingBot()

# Öffne eine Position (oder kommentiere diese Zeile aus für manuelles Trading)
bot.open_position()

# Starte die Überwachungsschleife
# Drücke Ctrl+C oder stoppe die Zelle, um den Bot zu beenden
bot.run()

# ==================================================================================
# Zelle 7: OPTIONAL - Manuelle Funktionen
# ==================================================================================

# Führe diese Zelle aus, um manuell Funktionen aufzurufen

# Zeige aktuelle offene Orders auf MEXC
def show_open_orders():
    orders = get_open_orders(SYMBOL)
    print(f"\n📊 Offene Orders für {SYMBOL}: {len(orders)}")
    for order in orders:
        print(f"   {order['side'].upper()} | {order['type']} | "
              f"Preis: {order['price']:.6f} | Menge: {order['amount']:.6f} | "
              f"Status: {order['status']}")

# Zeige aktuelles Guthaben
def show_balances():
    balance = exchange.fetch_balance()
    print("\n💰 Guthaben:")
    print(f"   USDT: {balance['free'].get('USDT', 0):.2f}")
    print(f"   UCN: {balance['free'].get('UCN', 0):.6f}")

# Storniere alle offenen Orders
def cancel_all_orders():
    orders = get_open_orders(SYMBOL)
    print(f"\n🗑️ Storniere {len(orders)} Orders...")
    for order in orders:
        cancel_order(SYMBOL, order['id'])

# Beispiel-Aufrufe (entferne das # um sie auszuführen):
# show_open_orders()
# show_balances()
# cancel_all_orders()

print("✅ Manuelle Funktionen bereit!")
print("   Entferne das # vor den Funktionen unten, um sie auszuführen")

✅ Alle Bibliotheken erfolgreich importiert!
✅ Konfiguration geladen!
📊 Symbol: UCN/USDT
💰 Position Size: 100 USDT
📈 Order Type: LIMIT
🎯 Gewinnziel: 2.0%
🛑 Stop-Loss: 1.0%
✅ Verbindung zu MEXC erfolgreich!
💵 USDT Guthaben: 0.13
💹 Aktueller UCN/USDT Preis: 1613.0700
✅ Hilfsfunktionen definiert!
✅ Trading Bot Klasse definiert!

🔍 Öffne neue Position...
💹 Aktueller Marktpreis: 1613.070000
🎯 Limit-Kaufpreis: 1611.456930 (-0.1%)
❌ Fehler beim Kaufen: mexc {"msg":"Insufficient position","code":30004}

🚀 MEXC TRADING BOT GESTARTET
📊 Order-Typ: LIMIT
🎯 Auto-Verkauf: AN
⏱️ Check-Interval: 10s


🤖 TRADING BOT STATUS | 23:22:57
💹 Aktueller UCN/USDT Preis: 1613.070000
📊 Aktive Positionen: 0/1
💵 USDT Guthaben: 0.13

⏳ Überwache Positionen... [23:23:07]
⏳ Überwache Positionen... [23:23:17]

🤖 TRADING BOT STATUS | 23:23:27
💹 Aktueller UCN/USDT Preis: 1613.070000
📊 Aktive Positionen: 0/1
💵 USDT Guthaben: 0.13

⏳ Überwache Positionen... [23:23:38]
⏳ Überwache Positionen... [23:23:48]

🤖 TRADING BOT STAT